# Práctica ML y DL - 01 Preprocesamiento
En este notebook realizaremos el preprocesamiento de los datos proporcionados por la empresa *Se nos fue de las Manos*. 
Tomaremos las mejores ideas de los proyectos de referencia:
- Limpieza de la variable `ALTITUD` (rangos a media) y su imputación por medias de estación.
- Imputación de la variable `SUPERFICIE` utilizando los datos históricos por finca.
- Agregación de las variables meteorológicas de `METEO` y `ETO` para complementar la información de cada campaña.

In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar los datos
df_train = pd.read_csv('data/TRAIN.txt', sep='|', encoding='utf-8')
print("Dimensiones TRAIN:", df_train.shape)
df_train.head()

Dimensiones TRAIN: (8526, 11)


,CAMPAÑA,ID_FINCA,ID_ZONA,ID_ESTACION,ALTITUD,VARIEDAD,MODO,TIPO,COLOR,SUPERFICIE,PRODUCCION
0,14,76953,515,4,660,26,2,0,1,0.0,22215.0
1,14,84318,515,4,660,26,2,0,1,0.0,22215.0
2,14,85579,340,4,520,32,2,0,1,0.0,20978.0
3,14,69671,340,4,520,32,2,0,1,0.0,40722.0
4,14,14001,852,14,NaN,81,1,0,1,0.0,14126.0


## 1. Limpieza de ALTITUD
La variable `ALTITUD` a veces viene expresada como un rango (ej. "600-700"). Las transformaremos a la media del rango y la convertiremos a numérico.

In [9]:
def curar_altitud(x):
    if pd.isna(x):
        return np.nan
    x_str = str(x).strip()
    if '-' in x_str:
        partes = x_str.split('-')
        return (float(partes[0]) + float(partes[1])) / 2
    return float(x_str)

df_train['ALTITUD'] = df_train['ALTITUD'].apply(curar_altitud)

# Imputar ALTITUD faltante usando la media de su ID_ESTACION
altitud_medias = df_train.groupby('ID_ESTACION')['ALTITUD'].mean()
df_train['ALTITUD'] = df_train.apply(
    lambda row: altitud_medias[row['ID_ESTACION']] if pd.isna(row['ALTITUD']) else row['ALTITUD'], 
    axis=1
)

print("Nulos en ALTITUD:", df_train['ALTITUD'].isna().sum())

Nulos en ALTITUD: 0


## 2. Imputación de SUPERFICIE
La variable `SUPERFICIE` suele faltar en años anteriores a 2020. Como la finca habitualmente mantiene su superficie, vamos a heredar el valor más reciente de superficie que tenga esa misma finca.

In [ ]:
# Ordenar por campaña para imputar hacia atrás (bfill) y hacia adelante (ffill) por ID_FINCA
df_train.sort_values(by=['ID_FINCA', 'CAMPAÑA'], inplace=True)

# Convertimos 0 a NaN para imputarlo también si lo hubiera
df_train['SUPERFICIE'] = df_train['SUPERFICIE'].replace(0, np.nan)

# Imputar agrupando por finca
df_train['SUPERFICIE'] = df_train.groupby('ID_FINCA')['SUPERFICIE'].transform(lambda x: x.bfill().ffill())

# Si aún quedan fincas sin superficie, imputamos con la media global o por variedad (elegimos variedad)
superficie_medias = df_train.groupby('VARIEDAD')['SUPERFICIE'].mean()
df_train['SUPERFICIE'] = df_train.apply(
    lambda row: superficie_medias[row['VARIEDAD']] if pd.isna(row['SUPERFICIE']) else row['SUPERFICIE'],
    axis=1
)
df_train['SUPERFICIE'].fillna(df_train['SUPERFICIE'].mean(), inplace=True)
print("Nulos en SUPERFICIE:", df_train['SUPERFICIE'].isna().sum())


      CAMPAÑA  ID_FINCA  ID_ZONA  ID_ESTACION  ALTITUD  VARIEDAD  MODO  TIPO  \
671        14       200       86           12    482.5        59     1     0   
1819       15       200       86           12    482.5        59     1     0   
2915       16       200       86           12    482.5        59     1     0   
3970       17       200       86           12    482.5        59     1     0   
5037       18       200       86           12    482.5        59     1     0   
...       ...       ...      ...          ...      ...       ...   ...   ...   
3452       17     99693      899           16    640.0        81     1     0   
4471       18     99693      899           16    640.0        81     1     0   
5522       19     99693      899           16    640.0        81     1     0   
6568       20     99693      899           16    640.0        81     1     0   
7580       21     99693      899           16    640.0        81     1     0   

      COLOR  SUPERFICIE  PRODUCCION  
6

## 3. Variables temporales / Históricas
Vamos a calcular la producción media histórica de cada finca hasta el año anterior, para darle al modelo contexto sobre si la finca es de alta o baja producción.

In [11]:
df_train.sort_values(by=['ID_FINCA', 'CAMPAÑA'], inplace=True)

# Produccion media histórica (excluyendo la campaña actual para evitar data leakage)
df_train['PRODUCCION_HISTORICA_MEDIA'] = df_train.groupby('ID_FINCA')['PRODUCCION'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Para los NA (primer año de la finca), usamos la produccion media global hasta ese momento o 0
df_train['PRODUCCION_HISTORICA_MEDIA'].fillna(0, inplace=True)

df_train.head()

,CAMPAÑA,ID_FINCA,ID_ZONA,ID_ESTACION,ALTITUD,VARIEDAD,MODO,TIPO,COLOR,SUPERFICIE,PRODUCCION,PRODUCCION_HISTORICA_MEDIA
671,14,200,86,12,482.5,59,1,0,1,0.37,1900.000,0.000000
1819,15,200,86,12,482.5,59,1,0,1,0.37,778.104,1900.000000
2915,16,200,86,12,482.5,59,1,0,1,0.37,1636.200,1339.052000
3970,17,200,86,12,482.5,59,1,0,1,0.37,829.008,1438.101333
5037,18,200,86,12,482.5,59,1,0,1,0.37,607.212,1285.828000


## 4. Datos Meteorológicos
Vamos a agregar las variables de `METEO` a nivel de campaña y estación. Consideraremos el periodo relevante para la uva (por ejemplo, desde septiembre del año anterior hasta agosto de la campaña). Por simplicidad y rapidez, calcularemos estadísticos globales anuales.

In [12]:
df_meteo = pd.read_csv('data/METEO.txt', sep='|', encoding='utf-8')
df_meteo['validTimeUtc'] = pd.to_datetime(df_meteo['validTimeUtc'], errors='coerce')
df_meteo['YEAR'] = df_meteo['validTimeUtc'].dt.year
df_meteo['MONTH'] = df_meteo['validTimeUtc'].dt.month

# Asignamos la campaña meteorologica: de julio a junio (como en las soluciones de referencia)
# Si el mes es >= 7, pertenece a la campaña del año siguiente
df_meteo['CAMPAÑA_METEO'] = np.where(df_meteo['MONTH'] >= 7, df_meteo['YEAR'] + 1, df_meteo['YEAR'])
# Ajustamos las campañas al formato de TRAIN (14, 15, ..., 22)
df_meteo['CAMPAÑA_METEO'] = df_meteo['CAMPAÑA_METEO'] - 2000

# Agregamos por ID_ESTACION y CAMPAÑA_METEO
agregaciones = {
    'temperature': ['mean', 'max', 'min'],
    'precip24Hour': ['mean', 'sum'],
    'relativeHumidity': ['mean']
}

df_meteo_agg = df_meteo.groupby(['ID_ESTACION', 'CAMPAÑA_METEO']).agg(agregaciones).reset_index()
# Aplanar nombres de columnas
df_meteo_agg.columns = ['ID_ESTACION', 'CAMPAÑA_METEO'] +                        [f"{col[0]}_{col[1]}" for col in df_meteo_agg.columns[2:]]

# Hacemos merge con TRAIN
df_train = df_train.merge(
    df_meteo_agg, 
    left_on=['ID_ESTACION', 'CAMPAÑA'], 
    right_on=['ID_ESTACION', 'CAMPAÑA_METEO'], 
    how='left'
)
df_train.drop(columns=['CAMPAÑA_METEO'], inplace=True)

# Imputar posibles NA meteorologicos con medias globales de esa campaña
for col in df_meteo_agg.columns[2:]:
    medias_campana = df_train.groupby('CAMPAÑA')[col].transform('mean')
    df_train[col].fillna(medias_campana, inplace=True)
    # Por si queda alguno suelto
    df_train[col].fillna(df_train[col].mean(), inplace=True)

## 5. Guardado de datos preprocesados
Almacenamos los datos para poder utilizarlos en el notebook de modelado.

In [13]:
df_train.to_csv('data/processed_train_data.csv', index=False)
print("Datos preprocesados guardados en data/processed_train_data.csv")

Datos preprocesados guardados en data/processed_train_data.csv
